# NIOS PDF Extraction — Docling on Kaggle

Extracts text, equations, tables, and images from NIOS chapter PDFs using IBM's **Docling**.

## Setup
1. Add **`dipankaj/nios-chapter-pdfs`** as a dataset input (Kaggle → Input → Add Dataset)
2. Edit `TARGET_SUBJECTS` below if you want a subset (default: all)
3. Enable **T4 GPU** accelerator → **Run All**

## Input layout (dataset)
```
/kaggle/input/nios-chapter-pdfs/
  class10/<subject-id>/Chapter N.pdf
  class12/<subject-id>/Chapter N.pdf
```

## Output layout (working dir)
```
/kaggle/working/extracted/
  _progress.json              ← saved after every subject
  class10/<subject-id>/chapters/
    Chapter N.json            ← Docling JSON (text + tables + figure refs)
    Chapter N_image_1.png     ← extracted figures
    _manifest.json
  class12/<subject-id>/chapters/
    ...
```


In [ ]:
# ── Config ── edit these if needed ───────────────────────────────────────────
from pathlib import Path
import json, time

# 'all' → every subject found in the dataset
# Single string → e.g. 'maths-12'
# List          → e.g. ['maths-12', 'business-10']
TARGET_SUBJECTS = 'all'

# Paths
DATASET_ROOT  = Path('/kaggle/input/nios-chapter-pdfs')
OUTPUT_ROOT   = Path('/kaggle/working/extracted')
PROGRESS_FILE = OUTPUT_ROOT / '_progress.json'

# Resume: skip chapters whose JSON already exists and is > 100 bytes
RESUME = True


In [ ]:
# ── Install Docling ───────────────────────────────────────────────────────────
!pip install -q --upgrade docling


In [ ]:
# ── Discover PDFs and build work plan ────────────────────────────────────────
import re

def chapter_sort_key(pdf_path):
    """Sort Chapter 1, Chapter 2 ... numerically."""
    match = re.search(r'\d+', pdf_path.stem)
    return int(match.group()) if match else 999


def discover_all_subjects(dataset_root):
    """Return list of Path objects for every subject directory that contains PDFs."""
    subjects = []
    for class_dir in sorted(dataset_root.glob('class*')):
        if class_dir.is_dir():
            for subject_dir in sorted(class_dir.iterdir()):
                if subject_dir.is_dir() and list(subject_dir.glob('*.pdf')):
                    subjects.append(subject_dir)
    return subjects


def get_subject_name(subject_dir):
    """Read subject_name from _manifest.json, fallback to directory name."""
    manifest = subject_dir / '_manifest.json'
    if manifest.exists():
        try:
            return json.loads(manifest.read_text()).get('subject_name', subject_dir.name)
        except Exception:
            pass
    return subject_dir.name


def get_already_done(output_dir):
    """Return set of chapter stems already extracted (JSON > 100 bytes)."""
    if not output_dir.exists():
        return set()
    return {
        f.stem for f in output_dir.glob('*.json')
        if f.name != '_manifest.json' and f.stat().st_size > 100
    }


# ── Load progress tracker ─────────────────────────────────────────────────────
if PROGRESS_FILE.exists():
    progress = json.loads(PROGRESS_FILE.read_text())
    print(f'📂 Progress loaded — {len(progress["subjects"])} subject(s) tracked')
else:
    progress = {'last_updated': None, 'subjects': {}}
    print('📂 No existing progress — starting fresh')


# ── Filter to TARGET_SUBJECTS ─────────────────────────────────────────────────
all_subject_dirs = discover_all_subjects(DATASET_ROOT)

if TARGET_SUBJECTS == 'all':
    selected_subject_dirs = all_subject_dirs
elif isinstance(TARGET_SUBJECTS, str):
    selected_subject_dirs = [d for d in all_subject_dirs if d.name == TARGET_SUBJECTS]
else:
    selected_subject_dirs = [d for d in all_subject_dirs if d.name in TARGET_SUBJECTS]


# ── Build work plan ───────────────────────────────────────────────────────────
work_plan = []  # list of dicts, one per subject

print(f'\nDataset : {DATASET_ROOT}  ({len(all_subject_dirs)} total subjects)\n')
print(f'{"Subject":<22}  {"PDFs":>4}  {"Done":>4}  {"Left":>4}  Status')
print('─' * 52)

for subject_dir in selected_subject_dirs:
    subject_id   = subject_dir.name
    class_level  = subject_id.rsplit('-', 1)[-1]     # e.g. '12' from 'maths-12'
    subject_name = get_subject_name(subject_dir)
    all_pdfs     = sorted(subject_dir.glob('*.pdf'), key=chapter_sort_key)
    output_dir   = OUTPUT_ROOT / f'class{class_level}' / subject_id / 'chapters'
    done_stems   = get_already_done(output_dir) if RESUME else set()
    pending_pdfs = [p for p in all_pdfs if p.stem not in done_stems]

    work_plan.append({
        'subject_id':   subject_id,
        'subject_name': subject_name,
        'class_level':  class_level,
        'subject_dir':  subject_dir,
        'output_dir':   output_dir,
        'all_pdfs':     all_pdfs,
        'done_stems':   done_stems,
        'pending_pdfs': pending_pdfs,
    })

    status_icon = '✅ done' if not pending_pdfs else ('⏳ partial' if done_stems else '🆕 new')
    print(f'  {subject_id:<22}  {len(all_pdfs):>4}  {len(done_stems):>4}  {len(pending_pdfs):>4}  {status_icon}')

total_pending = sum(len(s['pending_pdfs']) for s in work_plan)
print(f'\n  → {total_pending} chapter(s) to extract across {len(work_plan)} subject(s)')


In [ ]:
# ── Extract PDFs with Docling ────────────────────────────────────────────────
from docling.document_converter import DocumentConverter
import traceback

# Plain constructor — works with all Docling versions on Kaggle
converter = DocumentConverter()


def extract_single_pdf(pdf_path, output_dir):
    """
    Convert one PDF with Docling.
    Saves:
      <stem>.json           – Docling structured JSON
      <stem>_image_N.png    – extracted figures (if any)
    Returns: (success: bool, image_count: int)
    """
    try:
        result     = converter.convert(str(pdf_path))
        doc        = result.document
        json_path  = output_dir / f'{pdf_path.stem}.json'

        with open(json_path, 'w', encoding='utf-8') as fh:
            json.dump(doc.export_to_dict(), fh, indent=2, ensure_ascii=False)

        # Extract embedded pictures
        image_count = 0
        for element, _ in doc.iterate_items():
            pil_image = getattr(element, 'image', None)
            if pil_image is not None:
                image_count += 1
                img_path = output_dir / f'{pdf_path.stem}_image_{image_count}.png'
                pil_image.save(img_path)

        size_kb = json_path.stat().st_size // 1024
        return True, image_count

    except Exception as exc:
        traceback.print_exc()
        return False, 0


def save_progress_checkpoint(subject_entry, ok_count, fail_count, failed_chapter_names):
    """Write subject manifest + flush _progress.json to disk."""
    output_dir   = subject_entry['output_dir']
    subject_id   = subject_entry['subject_id']
    class_level  = subject_entry['class_level']
    subject_name = subject_entry['subject_name']
    all_pdfs     = subject_entry['all_pdfs']

    # Per-subject manifest
    chapter_files = sorted(
        [f for f in output_dir.glob('*.json') if f.name != '_manifest.json'],
        key=lambda p: chapter_sort_key(p)
    )
    manifest = {
        'subject_id':   subject_id,
        'subject_name': subject_name,
        'class_level':  class_level,
        'extracted_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'extractor':    'docling',
        'chapters': [
            {'name': f.stem, 'file': f.name, 'size_bytes': f.stat().st_size}
            for f in chapter_files
        ],
    }
    with open(output_dir / '_manifest.json', 'w') as fh:
        json.dump(manifest, fh, indent=2)

    # Global progress
    progress['subjects'][subject_id] = {
        'class_level':      class_level,
        'total_pdfs':       len(all_pdfs),
        'extracted':        ok_count,
        'failed':           fail_count,
        'failed_chapters':  failed_chapter_names,
        'status':          'complete' if fail_count == 0 and ok_count == len(all_pdfs) else 'partial',
        'last_run':         time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    }
    progress['last_updated'] = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
    PROGRESS_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(PROGRESS_FILE, 'w') as fh:
        json.dump(progress, fh, indent=2)


# ── Main extraction loop ──────────────────────────────────────────────────────
if total_pending == 0:
    print('✅ Nothing to do — all chapters already extracted!')
else:
    for subj_idx, subj in enumerate(work_plan, 1):
        subject_id   = subj['subject_id']
        pending_pdfs = subj['pending_pdfs']
        output_dir   = subj['output_dir']

        if not pending_pdfs:
            print(f'[{subj_idx}/{len(work_plan)}] ⏭️  {subject_id} — already complete')
            continue

        print(f'\n[{subj_idx}/{len(work_plan)}] 📚 {subject_id}'
              f'  ({len(pending_pdfs)} to extract, {len(subj["done_stems"])} already done)')
        output_dir.mkdir(parents=True, exist_ok=True)

        ok_count             = 0
        fail_count           = 0
        failed_chapter_names = []

        for ch_idx, pdf_path in enumerate(pending_pdfs, 1):
            print(f'    [{ch_idx:>2}/{len(pending_pdfs)}] ⚙️  {pdf_path.stem} ...', end=' ', flush=True)
            success, img_count = extract_single_pdf(pdf_path, output_dir)
            if success:
                ok_count += 1
                print(f'✅  ({img_count} image(s))')
            else:
                fail_count += 1
                failed_chapter_names.append(pdf_path.stem)
                print('❌')

        print(f'  → ✅ {ok_count}  ❌ {fail_count}')
        if failed_chapter_names:
            print(f'  → Failed: {failed_chapter_names}')

        save_progress_checkpoint(subj, ok_count, fail_count, failed_chapter_names)
        print(f'  💾 Checkpoint saved')

    print('\n🎉 Extraction complete!')


In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────
print(f'{"Subject":<22}  {"Total":>5}  {"Done":>5}  {"Fail":>5}  Status')
print('─' * 58)
for sid, info in progress['subjects'].items():
    icon = '✅' if info['status'] == 'complete' else '⚠️ '
    print(f"  {sid:<22}  {info['total_pdfs']:>5}  {info['extracted']:>5}  {info['failed']:>5}  {icon}")
print(f'\n📋 Progress file → {PROGRESS_FILE}')

# Show a sample of the first extracted JSON's structure
sample_files = sorted(OUTPUT_ROOT.glob('class*/*/chapters/Chapter *.json'))
if sample_files:
    sample = sample_files[0]
    with open(sample) as fh:
        sample_data = json.load(fh)
    print(f'\n🔍 Sample: {sample.relative_to(OUTPUT_ROOT)}')
    print(f'   Docling keys: {list(sample_data.keys())}')
    n_texts    = len(sample_data.get('texts', []))
    n_tables   = len(sample_data.get('tables', []))
    n_pictures = len(sample_data.get('pictures', []))
    print(f'   texts={n_texts}  tables={n_tables}  pictures={n_pictures}')
